In [ ]:
import folium
from folium import Choropleth, LayerControl, FeatureGroup
import geopandas as gpd
import pandas as pd

# Task 1

# Cargar el shapefile de distritos
gdf_distritos = gpd.read_file("data\shape_file\DISTRITOS.shp")

# Cargar el Excel descargado
df = pd.read_excel("data/listado_iiee.xlsx")

# Asegurar de que esté en EPSG:4326 para usar con Folium
gdf_distritos = gdf_distritos.to_crs(epsg=4326)

In [ ]:
# Convertir a minúsculas para evitar problemas con mayúsculas/minúsculas
df['nivel_lower'] = df['Nivel / Modalidad'].str.lower()

# Filtros más controlados
filtro_inicial = df['nivel_lower'].str.contains('inicial')
filtro_primaria = df['nivel_lower'].str.contains('primaria')
filtro_secundaria = df['nivel_lower'].str.contains('secundaria')

# Aplicar filtros
df_inicial = df[filtro_inicial]
df_primaria = df[filtro_primaria]
df_secundaria = df[filtro_secundaria]

In [ ]:
# Crear GeoDataFrames por nivel
gdf_inicial = gpd.GeoDataFrame(df_inicial, geometry=gpd.points_from_xy(df_inicial['Longitud'], df_inicial['Latitud']), crs="EPSG:4326")
gdf_primaria = gpd.GeoDataFrame(df_primaria, geometry=gpd.points_from_xy(df_primaria['Longitud'], df_primaria['Latitud']), crs="EPSG:4326")
gdf_secundaria = gpd.GeoDataFrame(df_secundaria, geometry=gpd.points_from_xy(df_secundaria['Longitud'], df_secundaria['Latitud']), crs="EPSG:4326")

# Contar por Ubigeo
conteo_inicial = gdf_inicial.groupby('Ubigeo').size().reset_index(name='inicial')
conteo_primaria = gdf_primaria.groupby('Ubigeo').size().reset_index(name='primaria')
conteo_secundaria = gdf_secundaria.groupby('Ubigeo').size().reset_index(name='secundaria')

In [ ]:
# Cambiar nombre IDDIST por Ubigeo para hacer el merge
gdf_distritos['Ubigeo'] = (gdf_distritos['IDDIST'].astype(str).str.zfill(2))

# Asegurar que los Ubigeos en conteos sean string
conteo_inicial['Ubigeo'] = conteo_inicial['Ubigeo'].astype(str)
conteo_primaria['Ubigeo'] = conteo_primaria['Ubigeo'].astype(str)
conteo_secundaria['Ubigeo'] = conteo_secundaria['Ubigeo'].astype(str)

In [ ]:
# Asegurar sistema de coordenadas compatible
gdf_distritos = gdf_distritos.to_crs(epsg=4326)

# Unir conteos al shapefile
gdf_mapa = gdf_distritos.merge(conteo_inicial, on='Ubigeo', how='left')
gdf_mapa = gdf_mapa.merge(conteo_primaria, on='Ubigeo', how='left')
gdf_mapa = gdf_mapa.merge(conteo_secundaria, on='Ubigeo', how='left')

# Rellenar NaN con 0
for col in ['inicial', 'primaria', 'secundaria']:
    gdf_mapa[col] = gdf_mapa[col].fillna(0)

In [ ]:
# Crear el mapa base centrado en Perú
m = folium.Map(location=[-9.19, -75.0152], zoom_start=6)

# Cargar datos geoespaciales (Asegúrate de que gdf_mapa ya contiene las columnas de conteo)
# Por ejemplo, gdf_mapa debe contener columnas 'inicial', 'primaria', 'secundaria', y 'geometry'

# Crear el Choropleth para los tres niveles educativos
folium.Choropleth(
    geo_data=gdf_mapa,
    name='choropleth_inicial',
    data=gdf_mapa,
    columns=['Ubigeo', 'inicial'],
    key_on='feature.properties.Ubigeo',  # Asegúrate que coincida con la propiedad de los distritos en el shapefile
    fill_color='YlGnBu',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Nivel Inicial',
).add_to(m)

folium.Choropleth(
    geo_data=gdf_mapa,
    name='choropleth_primaria',
    data=gdf_mapa,
    columns=['Ubigeo', 'primaria'],
    key_on='feature.properties.Ubigeo',
    fill_color='YlOrRd',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Nivel Primaria',
).add_to(m)

folium.Choropleth(
    geo_data=gdf_mapa,
    name='choropleth_secundaria',
    data=gdf_mapa,
    columns=['Ubigeo', 'secundaria'],
    key_on='feature.properties.Ubigeo',
    fill_color='BuPu',
    fill_opacity=0.7,
    line_opacity=0.2,
    legend_name='Nivel Secundaria',
).add_to(m)

# Añadir el control de capas
folium.LayerControl().add_to(m)

# Mostrar el mapa
m


In [ ]:
# Task 2

# Lista de dptos
dptos = ['HUANCAVELICA', 'AYACUCHO']

# Aplicar filtros de dptos
prim_hv_ay = df_primaria[df_primaria['Departamento'].str.upper().isin(dptos)].copy()
sec_hv_ay = df_secundaria[df_secundaria['Departamento'].str.upper().isin(dptos)].copy()

import geopandas as gpd
from shapely.geometry import Point

# Crear objetos GeoDataFrame con coordenadas
prim_hv_ay['geometry'] = prim_hv_ay.apply(lambda row: Point(row['Longitud'], row['Latitud']), axis=1)
sec_hv_ay['geometry'] = sec_hv_ay.apply(lambda row: Point(row['Longitud'], row['Latitud']), axis=1)

gdf_prim = gpd.GeoDataFrame(prim_hv_ay, geometry='geometry', crs='EPSG:4326')
gdf_sec = gpd.GeoDataFrame(sec_hv_ay, geometry='geometry', crs='EPSG:4326')


In [ ]:
# Reproyectar ambos a metros (UTM zona 18S para Perú)
print("Reproyectando a EPSG:32718...")
gdf_prim_proj = gdf_prim.to_crs(epsg=32718)
gdf_sec_proj = gdf_sec.to_crs(epsg=32718)
print("gdf_prim_proj CRS:", gdf_prim_proj.crs)
print("gdf_sec_proj CRS:", gdf_sec_proj.crs)

# Contar cuántas secundarias hay dentro de 5 km de cada primaria
print("Calculando número de secundarias cercanas (<= 5 km)...")
conteos = []
for idx, row in gdf_prim_proj.iterrows():
    distancia = gdf_sec_proj.distance(row.geometry)
    count = distancia[distancia <= 5000].count()
    conteos.append((idx, count))
    if idx % 100 == 0:
        print(f"Procesado {idx}: {count} secundarias")

conteos_df = pd.DataFrame(conteos, columns=['idx', 'num_secundarias'])
print("Conteos ejemplo:")
print(conteos_df.head())

gdf_prim_proj['num_secundarias'] = conteos_df.set_index('idx')['num_secundarias']
print("Distribución de secundarias cercanas:")
print(gdf_prim_proj['num_secundarias'].describe())

In [ ]:
# Identificar la primaria con más y menos secundarias cerca
prim_min = gdf_prim_proj.loc[gdf_prim_proj['num_secundarias'].idxmin()]
prim_max = gdf_prim_proj.loc[gdf_prim_proj['num_secundarias'].idxmax()]
print("Primaria con MENOS secundarias cerca:")
print(prim_min[['num_secundarias', 'geometry']])
print("Primaria con MÁS secundarias cerca:")
print(prim_max[['num_secundarias', 'geometry']])

# Filtrar secundarias cercanas a esos puntos
print("Filtrando secundarias cercanas a la más aislada...")
nearby_min = gdf_sec_proj[gdf_sec_proj.geometry.distance(prim_min.geometry) <= 5000]
print(f"Secundarias encontradas: {len(nearby_min)}")

print("Filtrando secundarias cercanas a la mejor conectada...")
nearby_max = gdf_sec_proj[gdf_sec_proj.geometry.distance(prim_max.geometry) <= 5000]
print(f"Secundarias encontradas: {len(nearby_max)}")

In [ ]:
# Volver a EPSG:4326 para usar en Folium
prim_min_latlon = gpd.GeoSeries([prim_min.geometry], crs=32718).to_crs(epsg=4326).iloc[0]
prim_max_latlon = gpd.GeoSeries([prim_max.geometry], crs=32718).to_crs(epsg=4326).iloc[0]
nearby_min_latlon = nearby_min.to_crs(epsg=4326)
nearby_max_latlon = nearby_max.to_crs(epsg=4326)

# Crear mapa en Folium para la más aislada
import folium
m = folium.Map(location=[prim_min_latlon.y, prim_min_latlon.x], zoom_start=13)

# Extraer el código modular de la escuela para crear el popup
codigo_primaria = prim_min['Código Modular'] if 'Código Modular' in prim_min else 'ID desconocido'
cantidad_secundarias = len(nearby_min)

# Popup personalizado
popup_text = f"Primaria con menos secundarias cercanas<br>Código: {codigo_primaria}<br>Secundarias cercanas: {cantidad_secundarias}"

# Marcador de primaria
folium.Marker(
    location=[prim_min_latlon.y, prim_min_latlon.x],
    popup=popup_text,
    icon=folium.Icon(color='red')
).add_to(m)

# Círculo de 5 km
folium.Circle(
    radius=5000,
    location=[prim_min_latlon.y, prim_min_latlon.x],
    color='red',
    fill=True,
    fill_opacity=0.2
).add_to(m)

# Secundarias cercanas
for _, row in nearby_min_latlon.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        icon=folium.Icon(color='blue', icon='graduation-cap', prefix='fa'),
        popup="Secundaria cercana"
    ).add_to(m)
m

In [ ]:
# Segundo mapa: primaria con más secundarias cercanas
m2 = folium.Map(location=[prim_max_latlon.y, prim_max_latlon.x], zoom_start=13)

# Extrae algún identificador, por ejemplo el código modular si lo tuvieras
codigo_primaria = prim_max['Código Modular'] if 'Código Modular' in prim_max else 'ID desconocido'
cantidad_secundarias = len(nearby_max)

# Popup personalizado
popup_text = f"Primaria con más secundarias cercanas<br>Código: {codigo_primaria}<br>Secundarias cercanas: {cantidad_secundarias}"

# Marcador de primaria
folium.Marker(
    location=[prim_max_latlon.y+0.0001, prim_max_latlon.x],
    popup=popup_text,
    icon=folium.Icon(color='green')
).add_to(m2)

# Círculo de 5 km
folium.Circle(
    radius=5000,
    location=[prim_max_latlon.y, prim_max_latlon.x],
    color='green',
    fill=True,
    fill_opacity=0.2
).add_to(m2)

# Secundarias cercanas
for _, row in nearby_max_latlon.iterrows():
    folium.Marker(
        location=[row.geometry.y, row.geometry.x],
        icon=folium.Icon(color='purple', icon='graduation-cap', prefix='fa'),
        popup="Secundaria cercana"
    ).add_to(m2)
m2


In [ ]:
# Caso 1 – Primaria con menos secundarias cercanas (zona rural):
# Se encuentra en un centro poblado en la Provincia de Huamanga, Ayacucho; no obstante, se encuentra muy alejado de la ciudad. La falta de cerreteras cercanas, la altura y la baja densidad poblacional explican la escasez de colegios secundarios en las cercanías.

# Caso 2 – Primaria con más secundarias cercanas (zona urbana):
# Ubicada en la ciudad de Ayacucho, donde la accesibilidad es mayor y existe una mayor densidad poblacional (la ciudad más poblada en ambas regiones). La infraestructura escolar está más concentrada, reflejando mejores condiciones de conectividad y demanda educativa.